In [1]:
import os
import math
import random

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
def read_structure(file_path):
  with open(file_path, 'r') as file:
    lines = file.readlines()

  clean_lines = []

  for line in lines:
    line = line.strip()

    if line != '':
      clean_lines.append(line)

  N = int(clean_lines[0])
  sizes = []

  for i in range(1, N + 1):
    sizes.append(int(clean_lines[i]))

  return sizes

In [3]:
class Layer:
  def __init__(self, input_size, output_size):
    self.input_size = input_size
    self.output_size = output_size
    self.weights = []
    self.biases = []

    for i in range(output_size):
      row = []

      for j in range(input_size):
        row.append(random.random())

      self.weights.append(row)
      self.biases.append(random.random())

In [4]:
class NeuralNetwork:
  def __init__(self, sizes):
    self.sizes = sizes
    self.layers = []

    for i in range(len(sizes) - 1):
      layer = Layer(sizes[i], sizes[i + 1])
      self.layers.append(layer)

  def sigmoid(self, x):
    if x < -100:
      return 0.0
    if x > 100:
      return 1.0

    return 1 / (1 + math.exp(-x))

  def feedforward(self, inputs):
    a = inputs

    for layer in self.layers:
      next_a = []

      for i in range(layer.output_size):
        z = layer.biases[i]

        for j in range(layer.input_size):
          z = z + layer.weights[i][j] * a[j]

        next_a.append(self.sigmoid(z))

      a = next_a

    return a

  def load_weights(self, file_path):
    with open(file_path, 'r') as file:
      lines = file.readlines()

    clean_lines = []

    for line in lines:
      line = line.strip()

      if line != '':
        clean_lines.append(line)

    k = 0

    for layer in self.layers:
      for i in range(layer.output_size):
        numbers = clean_lines[k].split(',')
        numbers = [float(x) for x in numbers]

        for j in range(layer.input_size):
          layer.weights[i][j] = numbers[j]

        layer.biases[i] = numbers[-1]
        k = k + 1

  def print_network(self):
    for layer_index in range(len(self.layers)):
      layer = self.layers[layer_index]
      print('Layer', layer_index + 1)

      for i in range(layer.output_size):
        print(' neuron', i + 1, 'weights =', layer.weights[i], 'bias =', layer.biases[i])

In [5]:
file_path = 'xor_network.txt'

if os.path.exists(file_path) == False:
  file_path = '/content/drive/MyDrive/data/xor_network.txt'

sizes = read_structure(file_path)

print('Network structure:', sizes)

Network structure: [2, 2, 1]


In [6]:
random_network = NeuralNetwork(sizes)

print('Random network:')
random_network.print_network()

Random network:
Layer 1
 neuron 1 weights = [0.8336258668707649, 0.30331999451913694] bias = 0.7090333149347009
 neuron 2 weights = [0.8321260612275095, 0.9946676455728246] bias = 0.3432379774569967
Layer 2
 neuron 1 weights = [0.579487901973969, 0.3876851159035807] bias = 0.2253705367354567


In [7]:
print('Random network output:')

inputs = [[0, 0], [0, 1], [1, 0], [1, 1]]

for x in inputs:
  output = random_network.feedforward(x)
  print(x, '=>', output)

Random network output:
[0, 0] => [0.6985753368361965]
[0, 1] => [0.7226242969336065]
[1, 0] => [0.730865368249411]
[1, 1] => [0.7453198843877324]


In [8]:
weight_path = 'xor_weights.txt'

if os.path.exists(weight_path) == False:
  weight_path = '/content/drive/MyDrive/data/xor_weights.txt'

xor_network = NeuralNetwork(sizes)
xor_network.load_weights(weight_path)

print('XOR network from text file:')
xor_network.print_network()

XOR network from text file:
Layer 1
 neuron 1 weights = [20.0, 20.0] bias = -10.0
 neuron 2 weights = [-20.0, -20.0] bias = 30.0
Layer 2
 neuron 1 weights = [20.0, 20.0] bias = -30.0


In [9]:
print('XOR result:')

inputs = [[0, 0], [0, 1], [1, 0], [1, 1]]

for x in inputs:
  output = xor_network.feedforward(x)

  if output[0] >= 0.5:
    y_pred = 1
  else:
    y_pred = 0

  print(x, '=> probability =', round(output[0], 6), ', predict =', y_pred)

XOR result:
[0, 0] => probability = 4.5e-05 , predict = 0
[0, 1] => probability = 0.999955 , predict = 1
[1, 0] => probability = 0.999955 , predict = 1
[1, 1] => probability = 4.5e-05 , predict = 0
